In [ ]:
# Imports
import math
import numpy as np
from scipy.signal import lti
from scipy.interpolate import interp2d
import matplotlib.pyplot as plt

In [ ]:
# Constants
M_TO_FT = 3.28084

W20_LIGHT_TURBULENCE_kts = 15 
W20_LIGHT_TURBULENCE_kts = 30 
W20_LIGHT_TURBULENCE_kts = 45 

In [ ]:
# given
wingspan_m = 3.96

pose = {'position_m':[150, 22, 457.2]}
velocity_m_per_s = np.array([1.2, 1.5, 0.12])

In [ ]:
wind_at_20ft_kts = 15

In [ ]:
# Inverse comments for random noise over time
sim_start_s = 0
sim_end_s = 10
dt_s = 0.01

t_span = [sim_start_s, sim_end_s]
t_s = np.arange(t_span[0], t_span[1] + dt_s/2, dt_s)
noise_u = 2*np.random.rand(t_s.shape[0]) - 1
noise_v = 2*np.random.rand(t_s.shape[0]) - 1
noise_w = 2*np.random.rand(t_s.shape[0]) - 1
noise_p = 2*np.random.rand(t_s.shape[0]) - 1
noise_q = 2*np.random.rand(t_s.shape[0]) - 1
noise_r = 2*np.random.rand(t_s.shape[0]) - 1


# t_s = np.array( [10] )
# noise_u = np.array( [0.02378972]  )
# noise_v = np.array( [0.10536685]  )
# noise_w = np.array( [0.99992383]  )
# noise_p = np.array( [0.92265746] )
# noise_q = np.array( [0.48527403] )
# noise_r = np.array( [0.85048805] )


The Von Karman model defines turbulent wind to be a WSS stochasctic process such that:

$$ Y_i(f) = H_i(f) * X_i(f) $$

Where:

$i,\ Wind\ Component\ ([u,v,w,p,q,r]) $

$X(f),\ Input:\ Random\ Noise$

$Y(f),\ Output:\ Turbulent\ Wind\ Rate$

$H(f),\ Transfer Function:\ Filter\ on\ Noise$

Examing Specifically the Linear-Longitudnal(u) component of Turbulence in the frequency domain:
$$ Y_u(s) = H_u(s) * X_u(s) $$

The Von Karman model further stipulates the following to be the respective transfer functions for each component of the turbulence wind rate: 


Linear-Longitudanal, u
$$ H_u(s) = \sigma_u  \sqrt{\frac{2L_u}{\pi V}} \cdot  \frac{1 + 0.25 \frac{L_u}{V}s}{1 + 1.357\frac{L_u}{V}s +0.1987 (\frac{L_u}{V})^2s^2}$$
Linear-Lateral, v
$$ H_v(s) = \sigma_v  \sqrt{\frac{L_v}{\pi V}} \cdot \frac{1 + 2.7478\frac{L_v}{V}s + 0.3398 (\frac{L_v}{V})^2 s^2}{1 + 2.9958\frac{L_v}{V}s + 1.9754(\frac{L_v}{V})^2s^2 + 0.1539(\frac{L_v}{V})^3s^3}$$
Linear-Vertical, w
$$ H_w(s) = \sigma_w  \sqrt{\frac{L_w}{\pi V}} \cdot \frac{1 + 2.7478\frac{L_w}{V}s + 0.3398 (\frac{L_w}{V})^2 s^2}{1 + 2.9958\frac{L_w}{V}s + 1.9754(\frac{L_w}{V})^2s^2 + 0.1539(\frac{L_w}{V})^3s^3}$$
Angular-Longitudanal, p
$$H_p(s) = \sigma_w \sqrt{\frac{0.8}{V}}\cdot\frac{\left(\frac{\pi}{4b}\right)^\frac{1}{6}}{L_w^\frac{1}{3}\left(1 + \left(\frac{4b}{\pi V}\right)s\right)}$$
Angular-Longitudanal, q
$$H_r(s) = \frac{\mp \frac{1}{V}s}{1 + \left(\frac{3b}{\pi V}\right)s} \cdot H_v(s)$$
Angular-Longitudanal, r
$$H_q(s) = \frac{\pm \frac{1}{V}s}{1 + \left(\frac{4b}{\pi V}\right)s} \cdot H_w(s)$$

Where:

$V, aircraft\ velocity$

$b, aircraft\ wingspan$

$ L_i, scale\ length$

$ \sigma_i, intensity$

Where again the Dryden model defines scale length and intensity in terms of the wind rate component and a function of the aircraft altitude and a relative characterization of the wind speed at 20ft above launch conditions, such that:

#### Altitude: \[ - , 1000ft\]

$$L_u = L_v = \frac{h}{\left(0.177 + 0.000823h\right)^{1.2}}$$
$$ L_w = h $$
$$ \sigma_w = 0.1W_{20} $$
$$ \sigma_u = \sigma_v = \frac{1}{(0.177 + 0.000823h)^{0.4}}$$

#### Altitude: \[1000ft, 2000ft\]

$$ L_{i} = interp(h, L_{i}(h=1000ft), L_{i}(h=2000ft))$$

![Mid/High Turbulence Intensity](img/MidHigh_Turbulence_Intensity.png)

#### Altitude: \[2000ft, - \]
$$ L_u = L_v = L_w = 2500 $$
$$ \sigma_u = \sigma_v = \sigma_w $$ 




In [ ]:
# Probability of Exceedance Table
# Figure 7 from p. 49 of MIL-F-8785C


poe_altitude_ft = np.array([500.0, 1750.0, 3750.0, 7500.0, 15000.0, 25000.0, 35000.0, 45000.0, 55000.0, 65000.0, 75000.0, 80000.0])
poe_index = np.array([1,2,3,4,5,6,7]) # 3: Light, 4: Moderate, 6: Severe
poe_table = np.array([
    [ 3.2,   2.2,   1.5,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0,  0.0,  0.0],
    [ 4.2,   3.6,   3.3,   1.6,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0,  0.0,  0.0],
    [ 6.6,   6.9,   7.4,   6.7,   4.6,   2.7,   0.4,   0.0,   0.0,   0.0,  0.0,  0.0],
    [ 8.6,   9.6,  10.6,  10.1,   8.0,   6.6,   5.0,   4.2,   2.7,   0.0,  0.0,  0.0],
    [11.8,  13.0,  16.0,  15.1,  11.6,   9.7,   8.1,   8.2,   7.9,   4.9,  3.2,  2.1],
    [15.6,  17.6,  23.0,  23.6,  22.1,  20.0,  16.0,  15.1,  12.1,   7.9,  6.2,  5.1],
    [18.7,  21.5,  28.4,  30.2,  30.7,  31.0,  25.2,  23.1,  17.5,  10.7,  8.4,  7.2]
])

poe_lookup = interp2d(poe_altitude_ft, poe_index, poe_table)

print(poe_lookup(2500, 2))

In [ ]:
altitude_m = pose['position_m'][2]
altitude_ft = altitude_m * M_TO_FT
wingspan_ft = wingspan_m * M_TO_FT 

# def get_scale_length_and_intensities(altitude_ft, wingspan_ft):

#     scale_length = dict('')
#     intensity = dict()

if altitude_ft < 1000:
    scale_length_longitude  = altitude_ft / (0.177 + 0.000823*altitude_ft)**1.2
    scale_length_lateral    = scale_length_longitude
    scale_length_vertical   = altitude_ft

    intensity_longitude = 0.1*wind_at_20ft_kts / (0.177 + 0.000823*altitude_ft)**0.4
    intensity_lateral   = intensity_longitude
    intensity_vertical  = 0.1*wind_at_20ft_kts

elif altitude_ft > 2000:
    scale_length_longitude  = 2500
    scale_length_lateral    = 2500
    scale_length_vertical   = 2500

    intensity_longitude     = poe_lookup(altitude_ft, 4.0)
    intensity_lateral       = intensity_longitude
    intensity_vertical      = intensity_longitude

else:
    low = 1000 / (0.177 + 0.000823*1000)**1.2
    high = 2500

    intensity_vert_low = 0.1*wind_at_20ft_kts
    intensity_long_low = 0.1*wind_at_20ft_kts / (0.177 + 0.000823*1000)**0.4

    intensity_high     = poe_lookup(2000.0, 4.0)[0]

    scale_length_longitude  = np.interp(altitude_ft, [1000.0, 2000.0], [low, high])
    scale_length_lateral    = np.interp(altitude_ft, [1000.0, 2000.0], [low, high])
    scale_length_vertical   = np.interp(altitude_ft, [1000.0, 2000.0], [1000., high])

    intensity_longitude     = np.interp(altitude_ft, [1000.0, 2000.0], [intensity_long_low, intensity_high])
    intensity_lateral       = np.interp(altitude_ft, [1000.0, 2000.0], [intensity_long_low, intensity_high])
    intensity_vertical      = np.interp(altitude_ft, [1000.0, 2000.0], [intensity_vert_low, intensity_high])



print('Scale Length, Longitude: %d' % scale_length_longitude)
print('Scale Length, Lateral: %d'   % scale_length_lateral)
print('Scale Length, Vertical: %d'  % scale_length_vertical)

print('Intensity, Longitude: %d' % intensity_longitude)
print('Intensity, Lateral: %d'   % intensity_lateral)
print('Intensity, Vertical: %d'  % intensity_vertical)


In [ ]:
# linear rate filter
# given
# - sccale lengths
# - intensities
# - Velocity

velocity_ft_per_s = velocity_m_per_s * M_TO_FT
velocity = np.linalg.norm(velocity_ft_per_s)

velocity_scale_longitude    = scale_length_longitude / velocity
velocity_scale_lateral      = scale_length_lateral / velocity
velocity_scale_vertical     = scale_length_vertical / velocity


# numerators denoted with 'b'
# denominators denoted with 'a'

b_u     = intensity_longitude * math.sqrt(2 * velocity_scale_longitude / math.pi)
b0_u    = b_u * 1
bs_u    = b_u * .25 * velocity_scale_longitude
a0_u    = 1
as_u    = 1.357 * velocity_scale_longitude
as2_u   = 0.1987 * velocity_scale_longitude**2

b_v     = intensity_lateral * math.sqrt(velocity_scale_lateral / math.pi)
b0_v    = b_v
bs_v    = b_v * 2.7478 * velocity_scale_lateral
bs2_v   = b_v * 0.3398 * velocity_scale_lateral**2
a0_v    = 1
as_v    = 2.9958 * velocity_scale_lateral
as2_v   = 1.9754 * velocity_scale_lateral ** 2
as3_v   = 0.1539 * velocity_scale_lateral ** 3

b_w     = intensity_vertical * math.sqrt(velocity_scale_vertical / math.pi)
b0_w    = b_w
bs_w    = b_w * 2.7478 * velocity_scale_vertical
bs2_w   = b_w * 0.3398 * velocity_scale_vertical**2
a0_w    = 1
as_w    = 2.9958 * velocity_scale_vertical
as2_w   = 1.9754 * velocity_scale_vertical ** 2
as3_w   = 0.1539 * velocity_scale_vertical ** 3

H_u = lti(  [bs_u, b0_u],
            [as2_u, as_u, a0_u])

H_v = lti( [bs2_v, bs_v, b0_v], 
           [as3_v, as2_v, as_v, a0_v])

H_w = lti( [bs2_w, bs_w, b0_w], 
           [as3_w, as2_w, as_w, a0_w])

print(H_u)
print(H_v)
print(H_w)

In [ ]:

turbulence_linear_rate = np.array([
                                    H_u.output(noise_u,	t_s)[1],
                                    H_v.output(noise_v, t_s)[1],
                                    H_w.output(noise_w, t_s)[1]])

In [ ]:
if t_s.size > 1:
    plt.plot(t_s, turbulence_linear_rate[0])

In [ ]:
if t_s.size > 1:
    plt.plot(t_s, turbulence_linear_rate[1])

In [ ]:
if t_s.size > 1:
    plt.plot(t_s, turbulence_linear_rate[2])

### Solving In Simulation



# Appendix



## Power Spectral Density


In [ ]:

circular_frequency = 1
spatial_frequency = circular_frequency / velocity

# Theoretical Power Spectral Densities (PSD)
psd_u = 2 * intensity_longitude**2 * scale_length_longitude / \
                (math.pi * velocity * (1 + (1.339 * scale_length_longitude * spatial_frequency) ** 2) ** (5/6))

psd_v = intensity_lateral**2 * scale_length_lateral * (1 + 8/3 * (1.339*scale_length_lateral * spatial_frequency)**2) / \
                (math.pi * velocity * (1 + (1.339 * scale_length_lateral * spatial_frequency) ** 2) ** (11/6))

psd_w = intensity_vertical**2 * scale_length_vertical * (1 + 8/3 * (1.339*scale_length_vertical * spatial_frequency) **2) / \
                (math.pi * velocity * (1 + (1.339 * scale_length_vertical * spatial_frequency) ** 2) ** (11/6))

psd_p = intensity_vertical**2 / (velocity * scale_length_vertical) * 0.8 * (math.pi * scale_length_vertical / (4 * wingspan_ft)) ** (1/3) / \
                (1 + (4 * wingspan_ft * spatial_frequency / math.pi) ** 2)

psd_q = -spatial_frequency**2 * psd_v / \
                (1 + (3 * wingspan_ft * spatial_frequency / math.pi) ** 2)

psd_r = spatial_frequency**2 * psd_w / \
                (1 + (4 * wingspan_ft * spatial_frequency / math.pi) ** 2)




print('Power Spectral Density, Phi_u: %d' % psd_u)
print('Power Spectral Density, Phi_v: %d' % psd_v)
print('Power Spectral Density, Phi_w: %d' % psd_w)
print('Power Spectral Density, Phi_p: %d' % psd_p)
print('Power Spectral Density, Phi_q: %d' % psd_q)
print('Power Spectral Density, Phi_r: %d' % psd_r)